In [1]:
#MANUAL APPROCH
#INPUT
h = 32
w = 32
d = 10

fh = 3
fw = 3
fd = 3
s = 1
p = 0
out_h = int((h - fh + 2*p) / s) + 1
out_w = int((w - fw + 2*p) / s) + 1
out_d = int((d - fd + 2*p) / s) + 1

print("After Convolution:")
print(out_h, out_w, out_d)

ph = 2
pw = 2
pd = 2
s2 = 2

pool_h = int(out_h / s2)
pool_w = int(out_w / s2)
pool_d = int(out_d / s2)

print("After Pooling:")
print(pool_h, pool_w, pool_d)
gap_h = 1
gap_w = 1
gap_d = pool_d

print("After Global Average Pooling:")
print(gap_h, gap_w, gap_d)

After Convolution:
30 30 8
After Pooling:
15 15 4
After Global Average Pooling:
1 1 4


In [3]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
path = "/content/drive/MyDrive/catsvsdogs/test"

data = []
labels = []

classes = os.listdir(path)
for i, cls in enumerate(classes):
    folder = os.path.join(path, cls)
    images = os.listdir(folder)

    temp = []
    for j in range(min(10, len(images))):
        img_path = os.path.join(folder, images[j])

        img = cv2.imread(img_path)
        img = cv2.resize(img, (64, 64))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        temp.append(img)

    if len(temp) == 10:
        data.append(temp)
        labels.append(i)
data = np.array(data)
data = data.reshape(-1, 1, 10, 64, 64)

X = torch.tensor(data, dtype=torch.float32)
y = torch.tensor(labels)

print("input shape:", X.shape)
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.conv = nn.Conv3d(1, 4, 3)
        self.pool = nn.MaxPool3d(2)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(4, len(classes))

    def forward(self, x):
        x = self.conv(x)
        print("after conv:", x.shape)

        x = self.pool(x)
        print("after pool:", x.shape)

        x = self.gap(x)
        print("after gap:", x.shape)

        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x


model = Net()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    opt.zero_grad()

    out = model(X)
    loss = loss_fn(out, y)

    loss.backward()
    opt.step()

    print("epoch:", epoch+1, "loss:", loss.item())

input shape: torch.Size([2, 1, 10, 64, 64])
after conv: torch.Size([2, 4, 8, 62, 62])
after pool: torch.Size([2, 4, 4, 31, 31])
after gap: torch.Size([2, 4, 1, 1, 1])
epoch: 1 loss: 18.867385864257812
after conv: torch.Size([2, 4, 8, 62, 62])
after pool: torch.Size([2, 4, 4, 31, 31])
after gap: torch.Size([2, 4, 1, 1, 1])
epoch: 2 loss: 16.685579299926758
after conv: torch.Size([2, 4, 8, 62, 62])
after pool: torch.Size([2, 4, 4, 31, 31])
after gap: torch.Size([2, 4, 1, 1, 1])
epoch: 3 loss: 14.503592491149902
